# 시계열 백본 비교 실험 (Colab GPU)

**StockMixer vs PatchTST vs Chronos (zero-shot)** 동일 데이터 공정 비교

| 모델 | 방식 | 역할 |
|------|------|------|
| StockMixer | MLP-Mixer, 경량 | 하한 베이스라인 |
| PatchTST | 패치 Transformer | 강력한 비교 모델 |
| Chronos | T5 백본, zero-shot | 핵심 연구 방법론 |

평가 지표: **Directional Accuracy**, Spearman Corr, MAE, RMSE

> 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q chronos-forecasting einops
!pip install -q h5py pyarrow tqdm

In [ ]:
import gc
import warnings
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from pathlib import Path

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

DRIVE_ROOT = Path('/content/drive/MyDrive/grad_project')
DATA_DIR   = DRIVE_ROOT / 'data'
SEQ_DIR    = DATA_DIR / 'sequences'
H5_PATH    = SEQ_DIR / 'sequences.h5'
MODEL_DIR  = DRIVE_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN    = 60
N_PATCHES  = 12
PATCH_LEN  = 5
N_FEATURES = 20
PRED_LEN   = 5

BATCH_SIZE = 512
N_EPOCHS   = 10
LR         = 1e-3
MAX_TRAIN  = 100_000
PATIENCE   = 3

assert H5_PATH.exists(), f'파일 없음: {H5_PATH}'
print(f'sequences.h5: {H5_PATH.stat().st_size/1e9:.2f} GB')

## 1. 데이터 로드

In [ ]:
class KospiH5Dataset(Dataset):
    def __init__(self, h5_path, split, max_samples=None):
        with h5py.File(h5_path, 'r') as f:
            n = f[split]['y_ret'].shape[0]
            if max_samples and n > max_samples:
                step = n // max_samples
                sl   = slice(0, n, step)
            else:
                sl = slice(None)
            self.X      = f[split]['X'][sl][:max_samples or n]
            self.X_flat = f[split]['X_flat'][sl][:max_samples or n]
            self.y_ret  = f[split]['y_ret'][sl][:max_samples or n].astype(np.float32)
            self.y_dir  = f[split]['y_dir'][sl][:max_samples or n].astype(np.int64)
        print(f'{split:5s}: {len(self.y_ret):>8,}개 | 상승: {self.y_dir.mean():.3f}')

    def __len__(self):
        return len(self.y_ret)

    def __getitem__(self, i):
        return (
            torch.from_numpy(self.X[i]),
            torch.from_numpy(self.X_flat[i]),
            torch.tensor(self.y_ret[i]),
            torch.tensor(self.y_dir[i]),
        )


print('데이터 로드 중...')
train_ds = KospiH5Dataset(H5_PATH, 'train', max_samples=MAX_TRAIN)
val_ds   = KospiH5Dataset(H5_PATH, 'val')
test_ds  = KospiH5Dataset(H5_PATH, 'test')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('완료')

## 2. 평가 지표

In [ ]:
def compute_metrics(y_true, y_pred):
    dir_acc  = ((y_true > 0) == (y_pred > 0)).mean()
    spear, _ = spearmanr(y_true, y_pred)
    mae      = np.abs(y_true - y_pred).mean()
    rmse     = np.sqrt(((y_true - y_pred) ** 2).mean())
    return {
        'dir_acc':  round(float(dir_acc), 4),
        'spearman': round(float(spear), 4),
        'mae':      round(float(mae), 6),
        'rmse':     round(float(rmse), 6),
    }

## 3. 모델 정의

In [ ]:
class StockMixer(nn.Module):
    """MLP-Mixer 스타일: time-mixing + feature-mixing 교차 적용"""
    def __init__(self, seq_len=60, n_features=20, d_model=128, n_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'norm1':     nn.LayerNorm(n_features),
                'time_mix':  nn.Sequential(
                    nn.Linear(seq_len, d_model), nn.GELU(), nn.Linear(d_model, seq_len)
                ),
                'norm2':     nn.LayerNorm(n_features),
                'feat_mix':  nn.Sequential(
                    nn.Linear(n_features, d_model), nn.GELU(), nn.Linear(d_model, n_features)
                ),
            }) for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(seq_len * n_features, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, x_patch, x_flat):
        x = x_flat
        for lyr in self.layers:
            r = x
            x = lyr['norm1'](x)
            x = r + lyr['time_mix'](x.transpose(1, 2)).transpose(1, 2)
            r = x
            x = r + lyr['feat_mix'](lyr['norm2'](x))
        return self.head(x).squeeze(-1)


class PatchTST(nn.Module):
    """패치 Transformer: 12개 패치(5일×20피처)를 토큰으로 사용"""
    def __init__(self, n_patches=12, patch_len=5, n_features=20,
                 d_model=128, nhead=4, n_layers=2, dropout=0.1):
        super().__init__()
        patch_dim        = patch_len * n_features
        self.patch_embed = nn.Linear(patch_dim, d_model)
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed   = nn.Parameter(torch.zeros(1, n_patches + 1, d_model))
        enc_layer        = nn.TransformerEncoderLayer(
            d_model, nhead, d_model * 4, dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)
        self.norm        = nn.LayerNorm(d_model)
        self.head        = nn.Linear(d_model, 1)

    def forward(self, x_patch, x_flat):
        B, P, L, C = x_patch.shape
        x   = self.patch_embed(x_patch.view(B, P, L * C))
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1) + self.pos_embed
        x   = self.norm(self.transformer(x))
        return self.head(x[:, 0]).squeeze(-1)


sm = StockMixer(SEQ_LEN, N_FEATURES)
pt = PatchTST(N_PATCHES, PATCH_LEN, N_FEATURES)
print(f'StockMixer 파라미터: {sum(p.numel() for p in sm.parameters()):,}')
print(f'PatchTST   파라미터: {sum(p.numel() for p in pt.parameters()):,}')
del sm, pt

## 4. 학습 함수

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for x_patch, x_flat, y_ret, _ in loader:
        x_patch, x_flat, y_ret = x_patch.to(device), x_flat.to(device), y_ret.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_patch, x_flat), y_ret)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    for x_patch, x_flat, y_ret, _ in loader:
        pred = model(x_patch.to(device), x_flat.to(device)).cpu().numpy()
        preds.append(pred)
        trues.append(y_ret.numpy())
    return compute_metrics(np.concatenate(trues), np.concatenate(preds))


def train_model(model, name, train_loader, val_loader, test_loader):
    model     = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    criterion = nn.MSELoss()

    best_mae, no_improve, best_state = float('inf'), 0, None
    history = []

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss  = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_m    = evaluate(model, val_loader, DEVICE)
        scheduler.step()
        history.append({'epoch': epoch, 'train_loss': tr_loss, **val_m})
        print(f'[{name}] ep{epoch:02d} | loss={tr_loss:.5f} | val_dir={val_m["dir_acc"]:.4f} | val_mae={val_m["mae"]:.5f}')

        if val_m['mae'] < best_mae:
            best_mae   = val_m['mae']
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    test_m = evaluate(model.to(DEVICE), test_loader, DEVICE)
    print(f'\n[{name}] Test: {test_m}\n')
    return model, test_m, history

## 5. StockMixer 학습

In [ ]:
%%time
sm_model, sm_metrics, sm_hist = train_model(
    StockMixer(SEQ_LEN, N_FEATURES, d_model=128, n_layers=2),
    'StockMixer', train_loader, val_loader, test_loader
)
torch.save(sm_model.state_dict(), MODEL_DIR / 'stockmixer_best.pt')
print('StockMixer 저장 완료')

## 6. PatchTST 학습

In [ ]:
%%time
gc.collect()
torch.cuda.empty_cache()

pt_model, pt_metrics, pt_hist = train_model(
    PatchTST(N_PATCHES, PATCH_LEN, N_FEATURES, d_model=128, nhead=4, n_layers=2),
    'PatchTST', train_loader, val_loader, test_loader
)
torch.save(pt_model.state_dict(), MODEL_DIR / 'patchtst_best.pt')
print('PatchTST 저장 완료')

## 7. Chronos Zero-shot 추론

In [ ]:
%%time
from chronos import ChronosPipeline

gc.collect()
torch.cuda.empty_cache()

pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=DEVICE,
    torch_dtype=torch.float32,
)
print('Chronos 모델 로드 완료')

# test 데이터의 Adj_Close (feature index 0) 추출
context_all = torch.from_numpy(test_ds.X_flat[:, :, 0].astype(np.float32))
y_true_all  = test_ds.y_ret

CHRONOS_BATCH = 256
all_pred = []

for i in tqdm(range(0, len(context_all), CHRONOS_BATCH), desc='Chronos 추론'):
    ctx = context_all[i:i + CHRONOS_BATCH]
    with torch.no_grad():
        forecast = pipeline.predict(ctx, prediction_length=PRED_LEN, num_samples=20)
    # forecast: (B, 20, 5) → median → (B, 5)
    median_pred = forecast.median(dim=1).values
    last_obs    = ctx[:, -1]
    pred_ret    = (median_pred[:, -1] - last_obs) / (last_obs.abs() + 1e-8)
    all_pred.append(pred_ret.cpu().numpy())

chronos_pred    = np.concatenate(all_pred)
chronos_metrics = compute_metrics(y_true_all, chronos_pred)
print(f'\n[Chronos zero-shot] Test: {chronos_metrics}')

## 8. 결과 비교 및 시각화

In [ ]:
results = pd.DataFrame({
    'StockMixer':          sm_metrics,
    'PatchTST':            pt_metrics,
    'Chronos (zero-shot)': chronos_metrics,
}).T

print('=' * 60)
print('백본 비교 실험 결과 (Test Set)')
print('=' * 60)
print(results.to_string())
print()
print(f'랜덤 베이스라인 Directional Accuracy: 0.5000')
print(f'최고 성능 백본: {results["dir_acc"].idxmax()}')

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, hist in [('StockMixer', sm_hist), ('PatchTST', pt_hist)]:
    ep = [h['epoch'] for h in hist]
    axes[0].plot(ep, [h['train_loss'] for h in hist], label=name, marker='o', ms=4)
    axes[1].plot(ep, [h['dir_acc']    for h in hist], label=name, marker='o', ms=4)
axes[0].set_title('Train Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].axhline(0.5, color='gray', ls='--', lw=1.2, label='랜덤(50%)')
axes[1].set_title('Val Directional Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

# 최종 비교 막대 그래프
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
models = results.index.tolist()
colors = ['steelblue', 'tomato', 'seagreen']

for ax, col, title in [
    (axes[0], 'dir_acc',  'Directional Accuracy'),
    (axes[1], 'spearman', 'Spearman Correlation'),
]:
    vals = results[col].values
    bars = ax.bar(models, vals, color=colors, alpha=0.85, edgecolor='k', lw=0.5)
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    if col == 'dir_acc':
        ax.axhline(0.5, color='red', lw=1.2, ls='--', label='랜덤(50%)')
        ax.legend()
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', fontsize=9)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=10)

plt.suptitle('백본 모델 비교 실험 결과', fontsize=13)
plt.tight_layout()
plt.show()

## 9. 결과 저장

In [ ]:
results.to_csv(DATA_DIR / 'backbone_results.csv', encoding='utf-8-sig')
print('저장 완료: backbone_results.csv')
print()
print('▶ 다음 단계 (9월): 멀티모달 아키텍처 통합')
print(f'  선정 백본: {results["dir_acc"].idxmax()}')
print('  이현동 팀원 뉴스 임베딩과 결합 방식 실험')
print('  Early Fusion / Cross-Attention / Modality-Specific Experts')